# Phase 2: Fine-Tuning Llama-3.2-3B (Standard QLoRA)

Because we are bypassing Unsloth, we will build our training pipeline using the native Hugging Face ecosystem. Here are the core libraries:

1. **Transformers & Accelerate:** The foundational libraries for downloading, loading, and running Large Language Models and their tokenizers on GPU hardware.
2. **BitsAndBytes:** This library is the "Q" in QLoRA (Quantized LoRA). It mathematically shrinks the massive 3-Billion parameter Llama model down to 4-bit precision so it fits comfortably inside our free 16GB T4 GPU.
3. **PEFT (Parameter-Efficient Fine-Tuning):** Instead of training all 3 Billion parameters (which would crash the GPU), PEFT freezes the original model and attaches a tiny "adapter" network. The model only updates this tiny adapter during training.
4. **TRL (Transformer Reinforcement Learning):** We will use its `SFTTrainer` (Supervised Fine-Tuning) class to manage the complex training loop, batching, and loss calculations automatically.

In [ ]:
# 1. Install standard Hugging Face ecosystem libraries
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.3 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from huggingface_hub import login
from google.colab import userdata

# 2. Authenticate with Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

### Initializing the Base Model & LoRA Adapters
First, we configure `BitsAndBytes` to load the model in 4-bit mode. We are using the `unsloth/Llama-3.2-3B-Instruct` repository because it hosts the exact same Meta weights but bypasses the annoying Meta license approval gate.



Once the base model is loaded and frozen in 4-bit, we use `prepare_model_for_kbit_training()` to stabilize it, and then attach our LoRA adapters using `get_peft_model()`.

In [ ]:
model_name = "unsloth/Llama-3.2-3B-Instruct"

# 1. Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading Base Model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16, # <--- THE FIX: Forces standard FP16 for the T4 GPU
    token=hf_token
)

# 2. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Inject LoRA Adapters
print("Injecting LoRA Adapters...")
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading Base Model in 4-bit...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Injecting LoRA Adapters...
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


### Structuring the Conversation
Llama-3 is an "Instruct" model, meaning it has been trained to read text in a specific conversational format with distinct roles (System, User, and Assistant). We need to map our Hugging Face dataset columns (`prompt`, `answer`, `label`) into this exact conversational structure so the model understands what it is reading.

In [ ]:
repo_id = "ShravSiddhpura/cybersec-slopsquatting-crag"
print(f"Loading dataset from {repo_id}...")
dataset = load_dataset(repo_id, split="train")

system_prompt = """

You are a cybersecurity AI. Analyze the user's coding prompt and the
provided answer. If the answer recommends a fake, hallucinated Python package, output 1.
If the answer is grounded and safe, output 0.

 """
def format_chat_template(examples):
    formatted_texts = []

    for prompt, answer, label in zip(examples['prompt'], examples['answer'], examples['label']):
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Prompt: {prompt}\n\nSuggested Answer: {answer}"},
            {"role": "assistant", "content": str(label)}
        ]

        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)

    return {"text": formatted_texts}

print("Formatting dataset...")
dataset = dataset.map(format_chat_template, batched=True)
print("Data is ready for the SFTTrainer!")

Loading dataset from ShravSiddhpura/cybersec-slopsquatting-crag...


README.md:   0%|          | 0.00/343 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/904 [00:00<?, ? examples/s]

Formatting dataset...


Map:   0%|          | 0/904 [00:00<?, ? examples/s]

Data is ready for the SFTTrainer!


### Executing Supervised Fine-Tuning
The `SFTTrainer` handles all the heavy mathematical lifting (calculating loss, backpropagation, and updating the LoRA weights). We set `max_steps = 60` for a quick initial test run to ensure the training loss is decreasing properly.

In [ ]:
# 1. Strip the old columns so the trainer doesn't get confused
clean_dataset = dataset.remove_columns(["prompt", "answer", "label"])

# 2. Initialize the Trainer
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=clean_dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=2048,
        output_dir="./lora_results",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,#  Explicitly block BFloat16 in the config
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
    ),
)

# 3. THE FIX: Force the LoRA adapters back to Float32
# (This physically overrides TRL's buggy BFloat16 auto-casting)
import torch
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("Starting training...")
trainer.train()
print("Training Complete!")

Starting training...


Step,Training Loss
1,1.615178
2,1.543584
3,1.638284
4,1.472420
5,1.516618
6,1.395521
7,1.318743
8,1.150062
9,1.283776
10,1.147319


Training Complete!


In [ ]:
# ---> IMPORTANT: Replace with your actual username and desired model name <---
new_model_id = "ShravSiddhpura/Llama-3.2-3B-Cybersec-Slopsquatting"

print(f"Pushing LoRA adapters to {new_model_id}...")

# 1. Push the newly trained adapter weights (Updated to use 'token')
model.push_to_hub(new_model_id, token=hf_token)

# 2. Push the tokenizer (Updated to use 'token')
tokenizer.push_to_hub(new_model_id, token=hf_token)

print("SUCCESS! Your fine-tuned model is permanently saved on Hugging Face.")

Pushing LoRA adapters to ShravSiddhpura/Llama-3.2-3B-Cybersec-Slopsquatting...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9w2hmawq/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

SUCCESS! Your fine-tuned model is permanently saved on Hugging Face.
